# HybridSense — Korelasyon & Model Değerlendirme Analizi
**Kadir Has Üniversitesi | FENS 401-402 | Grup 04**  
Çağla Özal · Sena Berra Soydugan · Başak Yıldız

---
### v2 Değişikliği — Label Leakage Düzeltmesi
`v1`'de `supply_temp_C`, `delta_T_abs`, `supply_airflow_m3h` gibi HVAC kontrol sinyalleri  
feature olarak kullanılıyordu. `occupancy_label` da bu sinyallerden türetildiği için  
model doluluk yerine HVAC operasyon modunu öğreniyordu → **AUC = 0.998 (trivial solution)**.

`v2`'de yalnızca fiziksel olarak her ortamda geçerli 9 feature kullanılıyor:  
`return_temp_C`, `supply_humidity_pct`, `humidity_diff`, `temp_roc`, `return_dp_Pa`,  
`hour_sin`, `hour_cos`, `is_workday`, `is_workhour`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, classification_report,
    precision_score, recall_score
)
import warnings
warnings.filterwarnings("ignore")

# Stil
plt.rcParams.update({"figure.facecolor":"#0f1117","axes.facecolor":"#1a1d2e",
                     "axes.edgecolor":"#252840","text.color":"#F0F5FA",
                     "axes.labelcolor":"#7A8FA6","xtick.color":"#F0F5FA",
                     "ytick.color":"#F0F5FA","grid.color":"#252840"})
TEAL="#00B4D8"; ORG="#FF6B35"; GRN="#4CAF50"; RED="#EF5350"; WHT="#F0F5FA"

print("Kütüphaneler yüklendi.")

In [ ]:
# Veriyi yükle
df = pd.read_csv("master_full_features.csv", parse_dates=["timestamp"])
print(f"Toplam satır: {len(df):,}")
print(f"Tarih: {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
print(f"\noccupancy_label dağılımı:")
print(df['occupancy_label'].value_counts().to_string())

In [ ]:
# ─── Feature Setleri ──────────────────────────────────────
# v1: leaky (HVAC kontrol sinyalleri dahil)
FEATURES_V1 = [
    "supply_temp_C","return_temp_C","supply_humidity_pct",
    "supply_dp_Pa","return_dp_Pa","delta_T","delta_T_abs",
    "temp_roc","dp_roc","temp_std_1h","temp_mean_1h",
    "dp_std_1h","dp_mean_1h","temp_mean_2h","dp_mean_2h",
    "hour_sin","hour_cos","dow_sin","dow_cos","month_sin","month_cos",
    "is_workday","is_workhour","is_morning","is_evening","is_night",
    "temp_lag1","dp_lag1","temp_lag4","dp_lag4"
]

# v2: temiz (domain-transferable)
FEATURES_V2 = [
    "return_temp_C",       # oda sıcaklığı — insan ısısıyla yükselir
    "supply_humidity_pct", # nem — insan nefesiyle artar  
    "humidity_diff",       # besleme/dönüş nem farkı
    "temp_roc",            # sıcaklık değişim hızı
    "return_dp_Pa",        # dönüş tarafı basıncı
    "hour_sin",            # saatin sinüs kodlaması
    "hour_cos",            # saatin kosinüs kodlaması
    "is_workday",          # iş günü
    "is_workhour",         # mesai saati (08-18)
]

# Hedef
df_model = df[df["occupancy_label"] != 0].copy()
df_model["y"] = (df_model["occupancy_label"] == 1).astype(int)
y = df_model["y"].values
split = int(len(df_model) * 0.8)

print(f"v1: {len(FEATURES_V1)} feature  |  v2: {len(FEATURES_V2)} feature")
print(f"Eğitim: {split:,}  |  Test: {len(df_model)-split:,}")

In [ ]:
# ─── Tek Değişkenli AUC — Leakage Kanıtı ─────────────────
results = []
for col in FEATURES_V1 + ["energy_proxy", "energy_whatif"]:
    if col not in df_model.columns: continue
    try:
        x = df_model[col].ffill().fillna(0).values
        auc = roc_auc_score(y, x)
        auc = max(auc, 1 - auc)
        results.append((col, auc))
    except: pass

results.sort(key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(10, 7), facecolor="#0f1117")
ax.set_facecolor("#1a1d2e")
cols_plot = [r[0] for r in results[:20]]
aucs_plot = [r[1] for r in results[:20]]
colors = [RED if a > 0.85 else (ORG if a > 0.70 else TEAL) for a in aucs_plot]
bars = ax.barh(cols_plot[::-1], aucs_plot[::-1], color=colors[::-1], alpha=0.85)
ax.axvline(0.85, color=RED, lw=2, ls="--", label="Leakage eşiği (AUC>0.85)", alpha=0.8)
ax.axvline(0.5, color=WHT, lw=1, ls=":", alpha=0.3)
ax.set_xlabel("Tek Değişkenli AUC (label ile)", color="#7A8FA6")
ax.set_title("Feature Başına Tek Değişkenli AUC\nKırmızı = Label Leakage Kanıtı", 
             color=WHT, fontsize=13, fontweight="bold")
ax.legend(facecolor="#252840", labelcolor=WHT, fontsize=10)
ax.set_xlim(0.4, 1.02)
for bar, auc in zip(bars, aucs_plot[::-1]):
    ax.text(auc + 0.005, bar.get_y() + bar.get_height()/2, 
            f"{auc:.3f}", va="center", fontsize=8.5, color=WHT)
plt.tight_layout()
plt.savefig("leakage_aucs.png", dpi=130, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("Kırmızı barlar doğrudan label sızdırıyor → bunlar v2'de çıkarıldı")

In [ ]:
# ─── Korelasyon Heatmap (v2 feature seti) ────────────────
cols_corr = FEATURES_V2 + ["occupancy_label"]
corr_data = df_model[cols_corr].ffill().fillna(0).corr()

fig, ax = plt.subplots(figsize=(9, 8), facecolor="#0f1117")
mask = np.zeros_like(corr_data, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr_data, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor="#0f1117",
            ax=ax, annot_kws={"size": 9},
            cbar_kws={"shrink": 0.8})
ax.set_title("Korelasyon Matrisi — v2 Feature Seti (9 temiz özellik + label)",
             color=WHT, fontsize=12, fontweight="bold", pad=12)
ax.set_facecolor("#1a1d2e")
plt.xticks(rotation=45, ha="right", color=WHT, fontsize=9)
plt.yticks(rotation=0, color=WHT, fontsize=9)
plt.tight_layout()
plt.savefig("correlation_v2.png", dpi=130, bbox_inches="tight", facecolor="#0f1117")
plt.show()

In [ ]:
# ─── Model Eğitimi ────────────────────────────────────────
X = df_model[FEATURES_V2].ffill().fillna(0).values
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_features="sqrt", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Logistic Regression
sc = StandardScaler().fit(X_train)
lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(sc.transform(X_train), y_train)

# Predictions
pred_rf = rf.predict(X_test)
prob_rf = rf.predict_proba(X_test)[:, 1]
pred_lr = lr.predict(sc.transform(X_test))
prob_lr = lr.predict_proba(sc.transform(X_test))[:, 1]

print("=== MODEL SONUÇLARI (v2 — 9 temiz feature) ===")
print(f"{'Model':<25} {'AUC':>7} {'F1':>7} {'Acc':>8} {'Precision':>10} {'Recall':>8}")
print("-"*70)
for name, pred, prob in [("Random Forest", pred_rf, prob_rf), ("Logistic Regression", pred_lr, prob_lr)]:
    print(f"  {name:<23} {roc_auc_score(y_test,prob):>7.3f} {f1_score(y_test,pred):>7.3f} "
          f"{accuracy_score(y_test,pred)*100:>7.1f}% "
          f"{precision_score(y_test,pred):>10.3f} {recall_score(y_test,pred):>8.3f}")

In [ ]:
# ─── Confusion Matrix (ISO: TP/FP/TN/FN) ─────────────────
# Hocanın slide 38'e göre: TP, FP, TN, FN terminolojisi
fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor="#0f1117")
fig.suptitle("Confusion Matrix — Sınıflandırma Sonuçları\n(Kaynak: ISO 7730 → Slide 38)",
             color=WHT, fontsize=13, fontweight="bold")

for ax, pred, name, color in zip(axes, [pred_rf, pred_lr],
                                  ["Random Forest", "Logistic Regression"],
                                  [TEAL, ORG]):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Boş (0)", "Dolu (1)"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}", color=WHT, fontsize=12, fontweight="bold")
    ax.set_facecolor("#1a1d2e")
    ax.xaxis.label.set_color(WHT)
    ax.yaxis.label.set_color(WHT)
    tn, fp, fn, tp = cm.ravel()
    ax.text(0.5, -0.18, f"TP={tp}  FP={fp}  TN={tn}  FN={fn}",
            transform=ax.transAxes, ha="center", color=WHT, fontsize=10)
    
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=130, bbox_inches="tight", facecolor="#0f1117")
plt.show()

In [ ]:
# ─── ROC Eğrisi (Slide 46-47'ye göre) ───────────────────
fig, ax = plt.subplots(figsize=(7, 6), facecolor="#0f1117")
ax.set_facecolor("#1a1d2e")

for prob, name, color in [(prob_rf,"Random Forest (v2)",TEAL),(prob_lr,"Logistic Regression (v2)",ORG)]:
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=ax, color=color, lw=2.5)

# Diagonal = random classifier (hocanın slide 47)
ax.plot([0,1],[0,1], "k--", lw=1.5, alpha=0.5, label="Rastgele sınıflandırıcı (AUC=0.5)")
ax.fill_between([0,1],[0,1],[0,1], alpha=0.05, color="white")
ax.set_xlabel("False Positive Rate", color="#7A8FA6", fontsize=11)
ax.set_ylabel("True Positive Rate", color="#7A8FA6", fontsize=11)
ax.set_title("ROC Eğrisi — v2 Modeller\nDiagonal = rastgele tahmin (slide 47)",
             color=WHT, fontsize=12, fontweight="bold")
ax.legend(facecolor="#252840", labelcolor=WHT, fontsize=10)
ax.grid(color="#252840", alpha=0.6)
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=130, bbox_inches="tight", facecolor="#0f1117")
plt.show()

In [ ]:
# ─── Precision / Recall / F1 (Slide 40-41) ──────────────
print("=== SINIFLANDIRMA RAPORU (v2 — Random Forest) ===")
print(classification_report(y_test, pred_rf, target_names=["Boş (0)", "Dolu (1)"]))
print("\n=== SINIFLANDIRMA RAPORU (v2 — Logistic Regression) ===")
print(classification_report(y_test, pred_lr, target_names=["Boş (0)", "Dolu (1)"]))

# Görsel karşılaştırma
metrics = ["Precision", "Recall", "F1-Score", "AUC-ROC"]
rf_vals = [precision_score(y_test,pred_rf), recall_score(y_test,pred_rf),
           f1_score(y_test,pred_rf), roc_auc_score(y_test,prob_rf)]
lr_vals = [precision_score(y_test,pred_lr), recall_score(y_test,pred_lr),
           f1_score(y_test,pred_lr), roc_auc_score(y_test,prob_lr)]

x = np.arange(len(metrics))
fig, ax = plt.subplots(figsize=(9, 5), facecolor="#0f1117")
ax.set_facecolor("#1a1d2e")
ax.bar(x - 0.2, rf_vals, 0.35, label="Random Forest", color=TEAL, alpha=0.85)
ax.bar(x + 0.2, lr_vals, 0.35, label="Logistic Regression", color=ORG, alpha=0.85)
for i, (rv, lv) in enumerate(zip(rf_vals, lr_vals)):
    ax.text(i-0.2, rv+0.01, f"{rv:.3f}", ha="center", color=WHT, fontsize=9, fontweight="bold")
    ax.text(i+0.2, lv+0.01, f"{lv:.3f}", ha="center", color=WHT, fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(metrics, color=WHT, fontsize=11)
ax.set_ylim(0, 1.1)
ax.axhline(1.0, color="#252840", lw=1, ls="--")
ax.set_title("Model Karşılaştırması — Precision / Recall / F1 / AUC\n(Slide 40-41 metriklerine göre)",
             color=WHT, fontsize=12, fontweight="bold")
ax.legend(facecolor="#252840", labelcolor=WHT, fontsize=10)
ax.grid(axis="y", color="#252840", alpha=0.6)
plt.tight_layout()
plt.savefig("metrics_comparison.png", dpi=130, bbox_inches="tight", facecolor="#0f1117")
plt.show()

In [ ]:
# ─── Feature Importance ──────────────────────────────────
fi = sorted(zip(FEATURES_V2, rf.feature_importances_*100), key=lambda x: -x[1])
fig, ax = plt.subplots(figsize=(9, 5), facecolor="#0f1117")
ax.set_facecolor("#1a1d2e")
feats = [f for f, _ in fi]
imps = [i for _, i in fi]
time_set = {"hour_sin","hour_cos","is_workday","is_workhour"}
cols_fi = [ORG if f in time_set else TEAL for f in feats]
bars = ax.barh(feats[::-1], imps[::-1], color=cols_fi[::-1], alpha=0.85)
for bar, imp in zip(bars, imps[::-1]):
    ax.text(imp+0.3, bar.get_y()+bar.get_height()/2, f"{imp:.1f}%",
            va="center", color=WHT, fontsize=9.5, fontweight="bold")
ax.set_xlabel("Feature Importance (%)", color="#7A8FA6")
ax.set_title("Random Forest Feature Importance — v2 (9 temiz feature)\n"
             "Turuncu = zaman sinyali | Mavi = insan/fizik sinyali",
             color=WHT, fontsize=12, fontweight="bold")
ax.grid(axis="x", color="#252840", alpha=0.6)
time_imp = sum(i for f,i in fi if f in time_set)
human_imp = sum(i for f,i in fi if f not in time_set)
ax.text(0.98, 0.05, f"Zaman: {time_imp:.1f}%\nİnsan/fizik: {human_imp:.1f}%",
        transform=ax.transAxes, ha="right", color=WHT, fontsize=10,
        bbox=dict(boxstyle="round", facecolor="#252840", alpha=0.8))
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=130, bbox_inches="tight", facecolor="#0f1117")
plt.show()

In [ ]:
# ─── v1 vs v2 Karşılaştırma ──────────────────────────────
# v1 modelini yeniden eğit (leaky features)
avail_v1 = [f for f in FEATURES_V1 if f in df_model.columns]
X1 = df_model[avail_v1].ffill().fillna(0).values
rf1 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf1.fit(X1[:split], y_train)
prob_rf1 = rf1.predict_proba(X1[split:])[:,1]
pred_rf1 = rf1.predict(X1[split:])

print("="*55)
print("v1 vs v2 KARŞILAŞTIRMA (Random Forest)")
print("="*55)
print(f"  {'':30} {'AUC':>7} {'F1':>7} {'Acc':>8}")
print(f"  {'-'*52}")
print(f"  {'v1 (leaky, 30 feature)':<30} {roc_auc_score(y_test,prob_rf1):>7.3f} "
      f"{f1_score(y_test,pred_rf1):>7.3f} {accuracy_score(y_test,pred_rf1)*100:>7.1f}%")
print(f"  {'v2 (temiz, 9 feature)':<30} {roc_auc_score(y_test,prob_rf):>7.3f} "
      f"{f1_score(y_test,pred_rf):>7.3f} {accuracy_score(y_test,pred_rf)*100:>7.1f}%")
print()
print("  v1 AUC yakın 1.0 = HVAC modunu ezberliyor (leakage)")
print("  v2 AUC = 0.78   = gerçek doluluk sinyali öğreniyor")
print("  v2 model Siemens demo odasına transfer edilebilir.")